# Lab 2 — Getting data in
### EP0418 · Topic 2 · 2 practical hours (Week 2)

**You need:** the printed Lab 2 sheet (Brightspace, Topic 2). The notebook fetches its
own files. Answer on the sheet or in the notebook's boxes.

---
Singapore's rain gauges report every five minutes. Today you meet that data in the
three ways it arrives:

- **CSV:** a saved table, sixty days, one row per day (Task 1);
- **JSON:** one five-minute snapshot of every gauge, boxes inside boxes (Task 2);
- **a live API:** you ask the service directly, and it may refuse or limit you (Task 2).

Then you **plan first** and use the loop to turn the JSON into a table (Task 3).

## How the cells work

- **Grey code cells:** run each one when you reach it, top to bottom. Click in it
  and press Shift+Enter.
- **Text boxes** have a bold label ending in a colon, like **My answers:**.
  Double-click, type, press Shift+Enter. Or write on the printed sheet.
- **Gemini's code** arrives as a **green cell**. Click **Accept and run** (bottom of
  the chat panel), then run the task's check cell.
- **Check cells** start with `# TODO(you):`, which says what to run first. Never
  change them: if Gemini shows red and green lines inside one, click **Cancel**
  (red and green lines inside Gemini's own green cell are fine). Leave lines like
  `s = keep("s")` as they are. If a check cell says *Not there yet* or *No ... yet*,
  type in the chat *call the result* and the name the task asks for, accept, and
  run the check cell again.
- **Gemini's lab rules** (*EP0418 lab rules*) are built in. If a box *Review custom
  instructions* appears, click **Use custom instruction**. The sliders icon beside
  **+** in the chat box should be blue; if it is grey, click it and choose *EP0418
  lab rules*. Keep *Learn mode* off.
- **If Gemini says *Daily limit reached*:** paste the same prompt into
  ChatGPT, Claude or Copilot in a new tab. It cannot see your notebook: copy its code
  into a new cell (**+ Code** at the top of the page) and run it, and paste any
  failed check or red error back into its chat. Do the quiz last to save your limit.
- **Prompts** in grey quote boxes start with **Rules:** (for example, do not drop
  rows; show the row count before and after), then **Task:**. Keep the Rules when
  you change a prompt. A prompt with no example is **zero-shot**; one example of the
  output you want makes it **one-shot**, several make it **few-shot**.
- **Checks** (the `check.*` lines under a task) are the **verification contract**.
  Do not edit them. In industry they are called **evals**.
- **A failing check** says what it found and what to try. Failing checks are normal
  while you work.

> **First: File → Save a copy in Drive, and work in your copy.** This page opened from GitHub, and changes here are not saved. If a box *Review custom instructions* appears while you save the copy, click **Make a copy**: it keeps the lab's short rules for the AI assistant. Colab opens your copy in a new tab with *Copy of* at the start of its name at the top left: work in that tab and close the old one. Reopen your copy from Google Drive next time.

> **Then run the grey code cell just below** (click in it and press Shift+Enter, or click the play button at its left). Colab may warn that the notebook was not authored by Google: click **Run anyway**. You should see nothing in red; lines like *fetched ...*, a row count or a short table are normal. That means the course files and tools are loaded. Every lab starts this way. From here on, **run every grey code cell when you reach it**, top to bottom. **To write in a text box** (a cell with a bold label ending in a colon), double-click it, type, then press Shift+Enter.

In [ ]:
import os, urllib.request
REPO = "https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/"


def fetch(*names):
    """Download any course file that is not beside the notebook yet."""
    for name in names:
        if os.path.exists(name):
            continue
        try:
            urllib.request.urlretrieve(REPO + name, name)
            print("fetched", name)
        except Exception:
            print(f"Could not download {name}. Upload it with the folder icon on the left.")
            from google.colab import files
            files.upload()


fetch("ep0418.py")
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from ep0418 import check, show


def keep(name):
    """The value Gemini's cell already made under this name, or ... if there is none yet."""
    global _NAMED
    v = globals().get(name, ...)
    if v is ... or v is None:
        print(f"Not there yet: nothing called {name}. Not run Gemini's green cell yet? Click Accept and "
              f"run on it, then run this cell again. Already ran it? Type in the same chat: call the "
              f"result {name}. Then Accept and run, and run this cell again.")
        _NAMED = True
        return ...
    return v


def filled(*values):
    """True once every TODO above has a value. Otherwise says what to do."""
    global _NAMED
    if any(v is ... or v is None for v in values):
        if not globals().get("_NAMED"):      # keep() has already said exactly what is missing
            print("Not there yet. Run Gemini's code first (click Accept and run on its green cell, "
                  "or paste its code into a new cell and run it), then run this cell again.")
        _NAMED = False
        return False
    _NAMED = False
    return True


def show_predict(data, unit):
    """Prediction League: run the newest predict that Gemini wrote and print its code to copy."""
    ran = globals().get("In", [])                                   # every cell you have run, oldest first
    mine = [c for c in ran if "def predict(" in c and "The baseline predicts" not in c and "show_predict" not in c]
    if not mine:
        print("Gemini's predict has not run yet. Find the cell Gemini added in Step 2 (it starts with")
        print("def predict, or with import lines just above it), click in it, press Shift+Enter,")
        print("then run this cell again."); return
    code = mine[-1].strip()
    for short, line in (("np.", "import numpy as np"), ("pd.", "import pandas as pd")):
        if short in code and line not in code:                     # the league page starts empty
            code = line + "\n" + code
    ns = {}
    exec(code, ns)
    print("Your predict says", round(ns["predict"](data.copy()), 1), unit)
    print()
    print("----- copy from the line below -----")
    print(code)
    print("----- to the line above -----")

import json
fetch("sg_rainfall_daily.csv", "rain_now.json")

## Task 1 — The CSV and the five questions (20 minutes)

*Topic 2B: five questions to ask of any data you have never seen.*

1. **Run the cell below.** It answers the five questions, one line each, then draws
   a chart.
2. **Read every line and the chart.**
3. **Write down** the strangest number in the box *Strangest number:* below: a number
   that does not fit with the others, or that you cannot explain yet.

**Watch first:** how each line of the cell below finds its answer, on a 6-day made-up
table (it plays slowly; the real file has 60 days). When it ends, run the cell.
![The five questions, worked on a small table](https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/img/lab02_five_questions.gif)

In [ ]:
rain = pd.read_csv("sg_rainfall_daily.csv", parse_dates=["date"])

print("1. Rows and columns:", rain.shape, list(rain.columns))
print("2. Period:", rain.date.min().date(), "to", rain.date.max().date(),
      "| largest gap between rows:", rain.date.diff().max().days, "day(s)")
print("3. Smallest and largest, each column:")
print(rain.describe().loc[["min", "max"]].round(2))
print("4. Blanks:", int(rain.isna().sum().sum()), "| days with zero rain everywhere:", int((rain.rain_mm_mean == 0).sum()))
print("5. Anything impossible? negative rain:", int((rain[["rain_mm_mean", "rain_mm_max"]] < 0).sum().sum()))
show.series(rain, col="rain_mm_mean", title="Average rain across all gauges, mm per day")

**Expected output** *(computed from the data file when this notebook was built)*:

- **Line 1:** 60 rows, four columns.
- **Line 2:** every day present; largest gap 1 day.
- **Line 3:** a small table of each column's smallest and largest value.
- **Line 4:** no blanks; 3 completely dry days.
- **Line 5:** no negative rain.

**What you should see.** Sixty days, no gaps, no blanks, no negative rain. So the
file is complete, but two things are worth a second look: how far the largest
`rain_mm_max` is above the largest `rain_mm_mean`, and why `stations_reporting` has
a smallest and a largest value at all, when the number of gauges never changes.

**Write it down** in the box below: the strangest number, and what you think caused it.

**Strangest number:**

*(Your answer box: double-click this cell. Type your answer here.)*

<details><summary><b>Answer guidance</b> (open after you have written yours)</summary>

Two good answers.

- **125.5 in `rain_mm_max`.** This column is the wettest single gauge each day. On
  7 July one gauge measured 125.5 mm, but the average over all 77 gauges was only
  13.0 mm. Heavy rain in Singapore often falls on one small area at a time: it pours
  on one part of the island while the rest gets little or none. So one gauge can read
  ten times the average.
- **`stations_reporting` is 76 on some days, 77 on others.** On those days one gauge
  sent no reading: it was offline. Its rain is missing, not zero. "Missing" and
  "zero" are different things; Lab 4 is about that difference.
</details>

## Task 2 — JSON and the live feed (20 minutes)

*Topic 2B: JSON is not a table, and a live API can refuse you or limit you.*

Task 2 has two parts: **2A**, open the saved file; **2B**, ask the live service.

### 2A · Open the saved file (`rain_now.json`)

The file is one five-minute snapshot, saved exactly as the service sent it.

1. **Run the cell below.** It prints the file's top-level boxes (the named groups of
   data inside it), then the first item in each.
2. **Compare** two numbers it prints: how many stations, and how many values. Are
   they equal?

In [ ]:
raw = json.load(open("rain_now.json"))["data"]

print("top-level boxes:", list(raw.keys()))
print("how many stations:", len(raw["stations"]), "| first one:", raw["stations"][0])
reading = raw["readings"][0]
print("the reading is from:", reading["timestamp"])
print("how many values in it:", len(reading["data"]), "| first three:", reading["data"][:3])

**What the cell above just opened**, box by box (it plays slowly, about a minute):
![JSON as boxes inside boxes: stations, readings, and the id they share](https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/img/lab02_json_boxes.gif)

**What each field means.** Read this before Task 3: the join only makes sense once you
know what each name in the file stands for.

| Where | Field | What it means | Example |
|---|---|---|---|
| `stations` | `id` | the gauge's code. A reading uses this code, not the name | `S218` |
| `stations` | `name` | where the gauge stands | Bukit Batok Street 34 |
| `stations` | `location` | latitude and longitude of the gauge | 1.365, 103.751 |
| `stations` | `deviceId` | the device's code; here always the same as `id` | `S218` |
| `readings[0]` | `timestamp` | when the reading was taken, Singapore time | 2026-05-26 09:05 |
| `readings[0]["data"]` | `stationId` | which gauge this value came from: the same code as a station's `id` | `S218` |
| `readings[0]["data"]` | `value` | rain at that gauge in the last 5 minutes, in mm | 0, 5.9 |
| top level | `readingType`, `readingUnit` | what was measured (5-minute rain total) and its unit (mm) | `mm` |

**What you should see.** Two lists stored apart: **stations** (id, name, location)
and **readings** (station id, value). A reading holds only a station id, not a name.
So to name the wettest gauge you must match each value to its station by id: that
is called a **join**. There are 77 stations but 75 values: two gauges sent nothing.
After the join those two must keep their row, with the rain left blank, not 0,
because "sent nothing" is not "no rain". Task 3 does this.

**See this reading on the map.** The animation below plays this same reading
slowly, in about a minute: where it rained, the wettest gauge against the mean, and
the two gauges that sent nothing. It also answers Task 1's strangest number.
![the same rain_now.json reading on the map, the last frame of the notebook's animation. 13 gauges saw rain, all in the south-west. The wettest gauge is 18 times the mean, because the mean shares all the rain out over the 75 values. The two gauges that sent nothing are left out of the mean, not counted as 0.](https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/img/lab02_one_moment.gif)

### 2B · Ask the live service (today's data)

Now ask the rain service itself for its latest reading. **Run the cell below once**
and read what it prints. It may work: it then shows today's gauges, so its count can
differ from the saved file's. Or it may answer 403 (refused) or 429 (too many
requests at once). All three are normal. Nothing later depends on this cell.

In [ ]:
try:
    req = urllib.request.Request("https://api-open.data.gov.sg/v2/real-time/api/rainfall",
                                 headers={"User-Agent": "EP0418 lab"})
    live = json.load(urllib.request.urlopen(req, timeout=20))["data"]
    vals = [x["value"] for x in live["readings"][0]["data"]]
    print("live:", live["readings"][0]["timestamp"], "|", len(vals), "gauges |",
          sum(v > 0 for v in vals), "reporting rain right now | wettest", max(vals), "mm")
except Exception as e:
    print("The service surprised you:", e)
    print("That is normal: a live service can refuse you, limit how often you ask, "
          "or change its answer. The rest of the lab uses the saved file.")

**What happened when you ran the cell above**, and its three possible answers:
![A request to the rain service and its three answers: 200, 403, 429](https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/img/lab02_api_call.gif)

## Task 3 — Plan first, then the loop (45 minutes)

*Topic 2A: plan, prompt, generate, verify, refine.*

**The goal:** one table, one row per gauge, with columns `name` and `rain_mm`:

| name | rain_mm |
|---|---|
| Bukit Batok Street 34 | 0.0 |
| Upper Changi Road North | 2.4 |
| ... | ... |

**Step 1. Predict, before you ask the AI.** From 2A: a reading has only a station id,
no name, and the 77 stations sent only 75 values. Write two short answers in the box
below:

1. How will you know which name goes with which value?
2. What should the 2 gauges that sent nothing show in your table?

**My plan:**

*(Your answer box: double-click this cell. Type your answer here.)*

**Step 2. Ask Gemini for its plan, no code, and compare.** Click the **spark icon**
(bottom centre) to open Gemini, start a new chat (**⋮** › **New chat** › **Yes**),
paste this and press Enter:

> **Rules:** I am new to this. Answer in three plain sentences, no headings, no bullet
> points, no code.
>
> **Task:** I have JSON from a rain service with two lists: "stations" (each has id
> and name) and "readings" (each has a timestamp and a list of stationId and value).
> How would you turn one reading into a table with one row per station, showing its
> name and its value? What could go wrong?

It should say: match the lists by station id, and watch for stations with no
reading. Compare it with your plan in Step 1. If the answer does not mention the
station id, type *how do you know which name goes with which value?* If it gives you
a long paragraph instead of three sentences, type *simpler, three sentences*.

**Step 3. Ask for the code, in the same chat, with your own last sentence.** Paste
this into the chat box, but **do not press Enter yet**:

> **Rules:** Do not drop any rows. Show the row count before and after. Explain in
> plain words what the code does.
>
> **Task:** `raw` is a Python dict with `raw["stations"]`, a list of dicts with keys
> `id` and `name`, and `raw["readings"][0]["data"]`, a list of dicts with keys
> `stationId` and `value`. Write pandas code that builds a DataFrame `table` with one
> row per station and two columns, `name` and `rain_mm`, matching each value to its
> station by id. One example row of the table: name = Bukit Batok Street 34, rain_mm =
> 0.0. A station with no value must ______.

In the chat box, replace **______** with your own words: what your plan in Step 1
says the 2 silent gauges should show. Then open *Check your sentence* below and
compare. If yours misses something, add it. Then press Enter.

**Zero-shot or one-shot?** (Video 2.) Step 2's prompt gives no example of the answer:
it is **zero-shot**, which is fine for a plan in plain words. Step 3's prompt shows
one example row of the table you want: it is **one-shot**, so Gemini copies its shape,
the column names and the kind of value. Gemini copies an example exactly, mistakes
included, so make sure yours is right.

<details><summary><b>Check your sentence</b> (open after you have written yours)</summary>

A complete sentence: *A station with no value must keep its row, with rain_mm left
blank (NaN), not 0.* It says two things:

- **keep its row**: otherwise the gauge disappears from the table;
- **left blank, not 0**: otherwise "sent nothing" turns into "no rain".

If yours says only one of the two, add the other before you press Enter.

You could also add a second example row for a silent gauge: *name = Kallang Practice
Track, rain_mm = NaN*. With two examples the prompt is **few-shot**, and the second
example shows the rule for silent gauges as well as the shape.
</details>

Gemini offers its code as a green cell.

**Step 4. Run it, then the check.** Click **Accept and run**. (If the code appears only in
the chat, with no green cell: copy it, click **+ Code** at the top of the page,
paste it and run it.) Then run the check
cell just below and read its last line. Many 0.0 values are normal: rain falls in
one part of the island at a time.

**Step 5. Only if a check fails:** tell Gemini, in the same chat, which check failed
and what it said, for example *rows_at_least failed: 75 rows, expected 77*. Click
**Accept and run**, then run the check cell again.

In [ ]:
# TODO(you): run Gemini's code first (its green cell, wherever it landed, or a new cell), so that `table` exists. Then run this cell.

# ---- verification contract — do not edit ----
if "table" not in globals() or table is None:
    print("No `table` yet. Accept and run Gemini's cell (or paste its code into a new cell and run it), then run this cell again. "
          "If you did, Gemini used another name: type in the chat 'call the result table', accept, and run this cell again.")
else:
    check.columns_present(table, ["name", "rain_mm"])
    check.rows_at_least(table, len(raw["stations"]))      # one row per station: a join that lost the silent gauges fails here
    check.values_in_range(table, "rain_mm", 0, 200)
    check.report()
    print("stations:", len(raw["stations"]), "| values in the reading:", len(raw["readings"][0]["data"]),
          "| rows in your table:", len(table), "| blank rain_mm:", int(table.rain_mm.isna().sum()),
          "| zero rain_mm:", int((table.rain_mm == 0).sum()))

**What the code had to do**, on four of the 77 stations: match by id, then the three
ways it can turn out (it plays slowly, about a minute):
![Joining stations and readings by id, and the three outcomes](https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/img/lab02_join.gif)

**Read the last line** (it starts `stations:`) and compare:

| | rows in your table | blank rain_mm | what it means |
|---|---|---|---|
| **A** | 77 | 2 | **Right.** The two silent gauges are there, marked unknown. |
| **B** | 75 | 0 | **Wrong:** the join dropped the two silent gauges. `rows_at_least` fails. |
| **C** | 77 | 0 | **Wrong, and no check catches it:** "no reading" became "no rain". |

The zero count (about 62) is real: most gauges were dry.

**If you got A:** go on to the hand check below.

**If you got B or C:** ask Gemini to fix it. In the same chat, type one sentence with your
numbers, for example:
- for **B**: *The table has 75 rows, but there are 77 stations. Keep every station,
  with rain_mm left as NaN when it has no reading. Keep everything else.*
- for **C**: *The table has 77 rows but 0 blanks. The two stations with no reading
  must have rain_mm left as NaN, not 0. Fix that and keep everything else.*

Click **Accept and run**, then run the check cell again: it should now say 77 rows
and 2 blanks.
Give the numbers, not just *it didn't work*: with the numbers, Gemini knows what to
fix; without them, it guesses.

**Check two rows by hand.** The checks test the shape and the range. They cannot tell
whether each value sits next to the right name. **Run the cell below.** It looks up
two stations (the first, and the wettest) in the raw JSON and in your table; each
line ends `match: True` or `match: False`. If a line says `match: False`, type in the same chat *The hand
check says match: False for* and the station name, then *; the join paired the wrong
value with that name. Match on the station id.* Accept, then run the check cell and
this cell again.

In [ ]:
if "table" in globals() and table is not None:
    readings = raw["readings"][0]["data"]
    names = {st["id"]: st["name"] for st in raw["stations"]}
    wettest_id = max(readings, key=lambda x: x["value"])["stationId"]
    for sid in [raw["stations"][0]["id"], wettest_id]:          # row 1: the first station; row 2: the wettest
        raw_value = [x["value"] for x in readings if x["stationId"] == sid]
        in_table = table.loc[table.name == names[sid], "rain_mm"].tolist()
        print(f"{names[sid]:28s} raw JSON says {raw_value}   your table says {in_table}   match: {raw_value == in_table}")
    print()
    print("Wettest ten gauges in this snapshot:")
    top = table.nlargest(10, "rain_mm")
    print(top.to_string(index=False))
    top.iloc[::-1].plot.barh(x="name", y="rain_mm", legend=False, figsize=(7, 4),
                             title="Rain in the last five minutes, mm"); plt.tight_layout(); plt.show()

**Write it down** in the box below: did both rows say `match: True`? How many gauges
had rain? Take *values in the reading* minus *zero rain_mm*, both on the check cell's
last line, for example 75 − 62 = 13.
Which gauge was wettest?

**Hand check and wettest gauge:**

*(Your answer box: double-click this cell. Type your answer here.)*

<details><summary><b>Answer guidance</b> (open after you have written yours)</summary>

Both rows must match. If not, the join paired a value with the wrong name: the
silent fault here: the code runs without an error, and no check catches it. In this snapshot (26 May, 9:05
in the morning) 13 gauges show rain, all in the south-west of the island: Jurong,
Clementi, Pasir Panjang. The wettest, Jurong West Street 73, read 5.9 mm in five minutes, while
62 gauges read 0. Rain was falling on one area, not the whole island. The two gauges
with no value must be blanks, not zeros.
</details>

---
## Predict: How much rain next week? (15 minutes)

*The Prediction League: one question per lab, settled by reality. Individual work: you enter on your own, under your own name.*

**The question:** Singapore's island-average rainfall in mm, summed over the target week. The target is the week of Monday 2026-11-02.

**How it works:** you enter a short function, `predict(history)`, not a number. When
entries close, the league runs your function once on the latest data. When the target
is over, the real value is compared with yours. League page (with the closing time):
https://stanleysp-eng.github.io/EP0418-labs/predict.html?lab=2

**Step 1.** **Run the cell below.** It prints the number from the simple rule,
*next week will be like last week (the last seven days, summed)*. That is the number to beat.

In [ ]:
fetch("sg_rainfall_daily.csv")
rain = pd.read_csv("sg_rainfall_daily.csv", parse_dates=["date"])

def predict(history):
    # the baseline: next week will be like last week
    last_week = history["rain_mm_mean"].tail(7)
    print("last 7 days:", last_week.round(1).tolist())
    return float(last_week.sum())

print("The baseline predicts", round(predict(rain), 1), "mm")

**What you should see:** *The baseline predicts 30.3 mm*.

**How the simple rule got that number**, step by step:
![The baseline: the last 7 days of rain, summed](https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/img/lab02_baseline.gif)

**Step 2.** **Ask Gemini to write your predict.** Start a new
chat (**⋮** › **New chat** › **Yes**), paste the prompt and press Enter.

> **Rules:** Put it in a new cell and do not change the cells above. Write only the function and any import lines it needs. Keep the name *predict* and its one input, *history*, a pandas table with the columns date, rain_mm_mean (island average, mm), rain_mm_max (wettest gauge, mm) and stations_reporting, oldest first. Return one number with float(...). **Task:** Write a predict(history) that estimates this: singapore's island-average rainfall in mm, summed over the target week. Do it in a way you think beats the simple rule (next week will be like last week (the last seven days, summed)). Explain the idea in one sentence.

Click **Accept and run** on Gemini's green cell. It prints nothing; it only defines the
function. If Gemini shows red and green lines inside the Step 1 cell, click **Cancel** and type *put it in a new cell*.

**Step 3.** **Run the cell below.**

In [ ]:
# TODO(you): run Gemini's green cell first (Step 2)
show_predict(rain, "mm")

**What you should see:** *Your predict says* and a number, usually between 5 and 80 mm, then your
code between two dashed lines. If it says *Gemini's predict has not run yet*, do what that message says.
If you see a red error, paste the last red line into Gemini with *fix this*, click
**Accept and run**, and run this cell again.

<details><summary><b>Using ChatGPT, Claude or another AI instead of Gemini?</b> Open this for a prompt to copy</summary>

That AI cannot see this notebook, so it needs a longer prompt that spells out every
rule the league checks. Use it in place of Steps 2 and 3:

1. **Copy the prompt** in the grey box below.
2. **Paste it** into the other AI's chat box. Do not send it yet.
3. **Write your idea.** In the line *My idea: WRITE YOUR IDEA HERE*, replace *WRITE YOUR IDEA HERE* with your idea, in plain words.
   Plain text is fine: no bold or italics needed. If you have no idea yet, delete the whole line: the AI then chooses one.
4. **Optional:** to let the AI look at the numbers first, download the data,
   https://stanleysp-eng.github.io/EP0418-labs/data/rain_daily.csv (the same file your
   function receives, updated every night), and attach it to the chat.
5. **Send it.** Copy the code the AI gives back, then do **Step 4** below: paste that
   code into the league page's **Code** box, in place of the code from Step 3.

```text
I am entering a prediction contest. Write one Python function for it.

The question: Singapore's island-average rainfall in mm, summed over the target week. The target is the week of Monday 2026-11-02 to Sunday 2026-11-08.
The simple rule to beat: next week will be like last week (the last seven days, summed).
My idea: WRITE YOUR IDEA HERE

The contest runs the function automatically, so follow these rules exactly:
1. Name it predict, with exactly one input called history:  def predict(history):
2. history is a pandas DataFrame that the contest passes in, oldest row first. Its column names are exactly: date, rain_mm_mean, rain_mm_max, stations_reporting. What they hold: date, rain_mm_mean (island average, mm), rain_mm_max (wettest gauge, mm) and stations_reporting. The date column is a pandas datetime, one row per day, with no time zone.
3. Do not read any file and do not ask for input. All the data is already in history. If I attach a data file, look at it only to choose the idea; the function must still take everything from history.
4. Return one number with return float(...). Not a list, a table, text, None or NaN.
5. Use only pandas, numpy and the Python standard library, and finish in under 30 seconds.
6. Give only the import lines and the function: no example data, no test code, nothing that calls predict.

After the code, explain the idea in one sentence.
```
</details>

**Step 4. Submit on the league page.**
1. **Copy your code** from the Step 3 output: everything between the two dashed lines.
2. **Open** https://stanleysp-eng.github.io/EP0418-labs/predict.html?lab=2 in a new tab and type your name as on the class list (the same name every week).
3. **Paste** it into the **Code** box: click in it, Ctrl+A, Ctrl+V (Cmd on a Mac).
4. **Click Run tests** (about 15 seconds the first time). **What you should see:** *All 4 cases ran* and four tabs: three past cases and *Your entry*. On *Your entry*, Actual says *not known yet*, and the number differs from Colab's because the page uses newer data: both are normal. A red *missed by* only means the simple rule did better on that case. A **red heading** instead of *All 4 cases ran* is an error: copy the red text under the case marked red (or under the heading), paste it into the same Gemini chat with *fix this*, click **Accept and run**, run Step 3 again, then paste and **Run tests** again.
5. **Click Submit.** *Submitted at ...* confirms it. Submit again any time before the close; the latest one counts.

**Write it down** (on the sheet, or double-click here): your function's number (Step 3),
the baseline (Step 1), and the one-sentence idea Gemini gave. After the target, the
real answer is on the league page.

**My number / the baseline / the idea:**

<details><summary><b>Answer guidance</b> (open after you have written yours)</summary>

There is no single right answer here. A good entry gives all three parts: your
function's number, the baseline (30.3 mm), and an idea that uses the data. Weeks in the file ranged from 2.7 mm to 73.6 mm, and 17.3 mm of last week's 30.3 mm fell on one day, 22 July, so one storm more or less swings a week a long way. Ideas that use this: average the last four weeks instead of one (27.2 mm), or blend last week with a typical week (the sixty-day average times seven is 34.6 mm).
The common weak answer is "Gemini wrote it". It says who wrote the code, not what the idea is. The past-case
tabs on the league page show whether the idea would have beaten the simple rule
lately; three cases are too few to prove it.
</details>

---
## Read before you run

An assistant was asked which rain gauge was wettest in the snapshot. It wrote the code below, which ran and printed a station name.

The code is switched off: every line starts with `#`, so it cannot run yet.

1. **Read it. Do not run it.** Answer questions 1 and 2 in the box under the code
   cell, or on the sheet.
2. **Ask Gemini to explain it.** Click in the code cell, press Ctrl+A then Ctrl+C
   (Cmd on a Mac). In Gemini, start a new chat (**⋮** › **New chat** › **Yes**),
   paste the code into the chat box, add *Explain this code in plain words.* and
   press Enter. If Gemini offers a green cell or red and green lines, click
   **Cancel**: this step only explains. Check whether the explanation points out the problem.
   Often it does not.
3. **Then run it.** Delete the `# ` at the start of each line below the first line,
   run the cell, and see whether you were right.

**Question 1.** Which row does `.iloc[0]` pick after the sort: the smallest value or the largest? What will it print on a snapshot where most gauges read zero?

**Question 2.** Predict what the code will print, and write what it should print: the wettest gauge you wrote down in Task 3.

<details><summary>Hint 1: where to look</summary>

Which way does `sort_values` sort unless you tell it otherwise: smallest first, or largest first?

</details>
<details><summary>Hint 2: closer to the answer</summary>

Smallest first. Row 0 after the sort is the driest gauge, not the wettest. The code runs, prints a real station name, and is wrong. Two rows checked by hand would show it.

![sort_values puts the smallest first, so .iloc[0] is the driest gauge](https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/img/lab02_sort_first.gif)

</details>

An explanation that reads well is not evidence the code is right. 

**Optional, to fix it:** in the same chat, type *Fix the problem in this code that
Hint 2 describes. Say in one sentence what you changed.* If that one change matches
Hint 2, click **Accept and run** (accepting is right here: you asked for the fix). If
it changed other things too, click **Cancel** and type *fix only that, nothing else*.
You should see: the gauge it prints is now a wet one, the same wettest gauge you found in Task 3, not one at 0.0 mm.

In [ ]:
# ---- Every line below this one is the code. Do steps 1 and 2 first. For step 3, delete the "# " at the start of each line below, then run the cell. ----
# wettest = table.sort_values("rain_mm").iloc[0]
# print("wettest gauge:", wettest["name"], wettest["rain_mm"], "mm")

**Answer to question 1:**

**Answer to question 2:**

**Did Gemini's explanation mention the fault? (yes / no, and the sentence):**

**Was I right, once I ran it?**

---
## Quiz yourself before you leave (5 minutes)

The checks tested the code. This quiz tests what you understood. In Gemini, start a new chat
(**⋮** › **New chat** › **Yes**), paste this into the chat box and press Enter:

> **Rules:** Ask one short question at a time, in plain words. Take the examples from the code and outputs in this notebook. Wait for my answer. Then say in one sentence whether I am right, and why. Do not give the next question until I answer.
>
> **Task:** Quiz me on this lab: five questions on these ideas: the three ways data arrives (CSV, JSON, live API) and what goes wrong with each; the five questions to ask of any new dataset, in order; why JSON is not a table and what joining two lists by id means; the five steps of the loop (plan, prompt, generate, verify, refine) and the step most people skip; why 'it ran without an error' is weak evidence, and what checking two rows by hand adds. After the fifth, give my score out of 5 and the one idea I should revise, in one line.

Gemini asks one question at a time in the chat panel; answer in the chat box, in
your own words, without looking back. Ignore the suggested replies Gemini shows. If you
use ChatGPT, Claude or Copilot instead, first delete the sentence *Take the examples from
the code and outputs in this notebook.*: they cannot see your notebook.

Then write your score and the idea to revise below. Your tutor asks for them.

**Quiz score:** ___ / 5

**The idea I should revise, in my own words:**

*(Your answer box: double-click this cell. Type your answer here.)*

---
## Before you leave

Write these on the last page of the Lab 2 sheet, or double-click this cell:

- the strangest number in the CSV, and what caused it;
- the API surprise you met, if any;
- did the code Gemini wrote do what its plan said (match by id, keep the silent gauges)?

**Mistake caught:** one line, *(what I expected · what was there · how I found it)*.
If the AI made no mistake today, use the read-before-you-run code: *I expected the wettest
gauge · it printed the driest · the sort was smallest first.*